In [ ]:
%%configure
{
  "defaultLakehouse": {
    "name": { "parameterName": "LakehouseName", "defaultValue": "AzureCostAnalyzer_LH" },
    "id": { "parameterName": "LakehouseId", "defaultValue": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx" },
    "workspaceId": { "parameterName": "WorkspaceId", "defaultValue": "yyyyyyyy-yyyy-yyyy-yyyy-yyyyyyyyyyyy" }
  }
}

# 05 · Gold · By Tag (WIDE, dynamic) + `dim_tag_key`

Builds **`gold_chargeback_by_tag`** — the single source for every **tag-driven** view (Chargeback
multi-tag grouping + Action Center governance & inline tagging) — plus **`dim_tag_key`**, the
tag universe.

**Output — resource-grain, dynamic WIDE**:
`YearMonth × SubAccountName × ServiceCategory × ServiceName × RegionName × ResourceGroupName ×
ResourceId × ResourceName × ResourceType × (one column per discovered tag key)` →
`EffectiveCost`, `BilledCost`, `TagCount`.

**Tag discovery — no hardcoded dimensions**:
- Tag keys are read straight from the FOCUS `Tags` JSON — whatever tags you actually use.
- Keys are **lower-cased + trimmed** so case variants collapse (`Environment`/`environment`/`ENV ` → `environment`); values are trimmed, empties dropped.
- **Every** discovered key becomes its own column (its value, or **`"Untagged"`** when the resource lacks that key). Not just the top-N — all of them.
- **One row per (resource, month)** → costs are counted **exactly once**, so the report/app can **group/filter by several tags at the same time with no double counting**.
- **`TagCount`** = number of tags on the resource (`0` = fully untagged) → governance / Action Center.

**`dim_tag_key`** (the tag universe) maps each key → its sanitized **PascalCase** column
(`cost center` → `CostCenter`) + a friendly **display** + **rank by cost**, so the app discovers the
(dynamic) tag columns at runtime and renders friendly names / builds its GROUP BY. It replaces the
old `gold_tag_keys` helper.

**Dependencies**: `focus_partitioned` (silver) with `Tags` JSON, `ResourceId`, `ResourceName`.

> Standardized to the validated ACA model: **WIDE** (was long `ChargebackDimension`/`ChargebackKey`),
> `"Untagged"` (was `(sin etiqueta)`), one column per key + `dim_tag_key`.


In [ ]:
# gold_chargeback_by_tag — RESOURCE-GRAIN, DYNAMIC *WIDE* tag table (one column per tag key)
# + dim_tag_key (the tag universe). Tag KEYS are discovered from the FOCUS `Tags` JSON
# (NO hardcoded dimensions), normalized (trim+lower) so casing variants collapse. EVERY discovered
# key becomes its own column (its value, or "Untagged" when the resource lacks that key). ONE row
# per resource-month => costs counted EXACTLY ONCE, so the report/app can group/filter by SEVERAL
# tags at a time with NO double counting. `dim_tag_key` maps each key -> its (dynamic) column,
# display, and rank so the app discovers the tag columns at runtime (replaces the old gold_tag_keys).
# mssparkutils / notebookutils are available globally in the Fabric runtime (no import needed)
import re
from datetime import date
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F

# Lower-casing keys can collapse case-variants onto the SAME key (Env/env -> env). Make the last
# value win instead of raising "Duplicate map key" when building the tag map.
spark.conf.set("spark.sql.mapKeyDedupPolicy", "LAST_WIN")

# Partition pruning: read last 12 months
lookback_date = date.today() - relativedelta(months=12)
year_filter, month_filter = lookback_date.year, lookback_date.month

UNTAGGED = "Untagged"

def _dflt(col, default):
    c = F.trim(col)
    return F.when(c.isNull() | (c == ""), F.lit(default)).otherwise(c)

usage = (spark.table("focus_partitioned")
    .where((F.col("ChargeCategory") == "Usage")
           & ((F.col("Year") > year_filter) | ((F.col("Year") == year_filter) & (F.col("Month") >= month_filter))))
    .withColumn("YearMonth", F.date_format("ChargePeriodStart", "yyyy-MM")))

# 1) Normalize Tags JSON -> map<key,value> (keys trim+lower, values trim, empties dropped).
_norm = usage.withColumn(
    "tagmap",
    F.map_from_entries(
        F.filter(
            F.transform(
                F.map_entries(F.from_json("Tags", "map<string,string>")),
                lambda e: F.struct(F.trim(F.lower(e["key"])).alias("key"), F.trim(e["value"]).alias("value")),
            ),
            lambda e: e["key"].isNotNull() & (e["key"] != "") & e["value"].isNotNull() & (e["value"] != ""),
        )
    ),
)

# 2) Discover EVERY tag key (ranked by cost) — this drives the dynamic columns.
_key_cost = (_norm.select("EffectiveCost", F.explode("tagmap").alias("k", "v"))
    .groupBy("k").agg(F.sum("EffectiveCost").alias("EffectiveCost"))
    .orderBy(F.desc("EffectiveCost")))
_keys = [(r["k"], float(r["EffectiveCost"])) for r in _key_cost.collect()]

# 3) Safe, readable PascalCase column name per key + friendly display. Dedupe CASE-INSENSITIVELY —
#    Spark resolves columns case-insensitively by default, so `Databricksenvironment` and
#    `DatabricksEnvironment` are the SAME column (this raised COLUMN_ALREADY_EXISTS). Also avoid
#    colliding with the fixed (non-tag) columns. Collisions get a numeric suffix.
def _colname(key):
    parts = [p for p in re.split(r"[^0-9a-z]+", key) if p]
    return "".join(p.capitalize() for p in parts) or "Tag"

_FIXED = ["YearMonth", "SubAccountName", "ServiceCategory", "ServiceName", "RegionName",
          "ResourceGroupName", "ResourceId", "ResourceName", "ResourceType",
          "EffectiveCost", "BilledCost", "TagCount"]
_used = {c.lower() for c in _FIXED}
TAG_MAP = []   # rows: (rank, original_key, column_name, display, cost)
for i, (k, c) in enumerate(_keys, start=1):
    base = _colname(k)
    col, n = base, 1
    while col.lower() in _used:
        n += 1
        col = f"{base}{n}"
    _used.add(col.lower())
    TAG_MAP.append((i, k, col, k.title(), c))

# 4) Build the WIDE resource-grain fact: one column per key (value or "Untagged") + TagCount.
_res_type = _dflt(F.regexp_extract(F.lower(F.col("ResourceId")), r"providers/([^/]+/[^/]+)", 1), "(unknown)")
_r = _norm.select(
    "YearMonth", "SubAccountName", "ServiceCategory", "ServiceName", "RegionName",
    _dflt(F.col("x_ResourceGroupName"), "(no resource group)").alias("ResourceGroupName"),
    _dflt(F.col("ResourceId"), "(no resource)").alias("ResourceId"),
    _dflt(F.col("ResourceName"), "(no resource)").alias("ResourceName"),
    _res_type.alias("ResourceType"),
    "EffectiveCost", "BilledCost",
    F.when(F.col("tagmap").isNull(), F.lit(0)).otherwise(F.size("tagmap")).alias("TagCount"),
    F.col("tagmap"),
)
for (_, k, col, _, _) in TAG_MAP:
    _r = _r.withColumn(col, F.coalesce(_r["tagmap"].getItem(k), F.lit(UNTAGGED)))

_tag_cols = [col for (_, _, col, _, _) in TAG_MAP]
_group = ["YearMonth", "SubAccountName", "ServiceCategory", "ServiceName", "RegionName",
          "ResourceGroupName", "ResourceId", "ResourceName", "ResourceType"] + _tag_cols
gold_chargeback_by_tag = (_r.groupBy(*_group)
    .agg(F.sum("EffectiveCost").alias("EffectiveCost"),
         F.sum("BilledCost").alias("BilledCost"),
         F.max("TagCount").alias("TagCount")))
gold_chargeback_by_tag.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_chargeback_by_tag")

# 5) dim_tag_key — maps each discovered key -> its (dynamic) column + display + rank; the app reads
#    this to know which tag columns exist and to render friendly names / build its GROUP BY.
dim_tag_key = spark.createDataFrame(
    [(rank, key, col, disp, cost) for (rank, key, col, disp, cost) in TAG_MAP],
    "Rank int, TagKey string, ColumnName string, TagKeyDisplay string, EffectiveCost double",
).select("TagKey", "TagKeyDisplay", "ColumnName", "Rank", "EffectiveCost")
dim_tag_key.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_tag_key")

print(f"✓ gold_chargeback_by_tag (WIDE, dynamic): {gold_chargeback_by_tag.count():,} rows, {len(_tag_cols)} tag columns")
print(f"  tag columns: {_tag_cols}")
print(f"✓ dim_tag_key (tag universe): {len(TAG_MAP)} keys")
dim_tag_key.orderBy("Rank").show(50, truncate=False)
